In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt     
from scipy.stats import geom
from scipy.optimize import minimize

In [2]:
import pandas as pd

# Define start and end dates
start = "1984-09-07"
end = "2022-04-22"

# Create daily date range
dates = pd.date_range(start=start, end=end, freq="D")

# Build DataFrame with full YYYYMMDD
df = pd.DataFrame({
    "date_yyyymmdd": dates.strftime("%Y%m%d"),
    "weekday": dates.strftime("%A")
})

df.head()


,date_yyyymmdd,weekday
0,19840907,Friday
1,19840908,Saturday
2,19840909,Sunday
3,19840910,Monday
4,19840911,Tuesday


In [3]:
df.shape

(13742, 2)

In [4]:
apple = pd.read_csv(r"D:\data\aapl.us.txt")
apple.head()

,<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
0,AAPL.US,D,19840907,0,0.10122,0.10246,0.10000,0.10122,97236149,0
1,AAPL.US,D,19840910,0,0.10122,0.10153,0.09878,0.10062,75471114,0
2,AAPL.US,D,19840911,0,0.10153,0.10428,0.10153,0.10246,177965367,0
3,AAPL.US,D,19840912,0,0.10246,0.10306,0.09938,0.09938,155467926,0
4,AAPL.US,D,19840913,0,0.10490,0.10520,0.10490,0.10490,242135546,0


In [5]:
apple.shape

(9484, 10)

In [6]:
apple['<DATE>'] = pd.to_datetime(apple['<DATE>'], format="%Y%m%d")
apple.dtypes
apple.head()

,<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
0,AAPL.US,D,1984-09-07,0,0.10122,0.10246,0.10000,0.10122,97236149,0
1,AAPL.US,D,1984-09-10,0,0.10122,0.10153,0.09878,0.10062,75471114,0
2,AAPL.US,D,1984-09-11,0,0.10153,0.10428,0.10153,0.10246,177965367,0
3,AAPL.US,D,1984-09-12,0,0.10246,0.10306,0.09938,0.09938,155467926,0
4,AAPL.US,D,1984-09-13,0,0.10490,0.10520,0.10490,0.10490,242135546,0


In [7]:
apple.columns = apple.columns.str.strip().str.strip("<>").str.upper()

In [8]:
apple['DayOfWeek'] = apple['DATE'].dt.day_name()
apple.tail()

,TICKER,PER,DATE,TIME,OPEN,HIGH,LOW,CLOSE,VOL,OPENINT,DayOfWeek
9479,AAPL.US,D,2022-04-18,0,163.92,166.5984,163.57,165.07,69023941,0,Monday
9480,AAPL.US,D,2022-04-19,0,165.02,167.8200,163.91,167.40,67723833,0,Tuesday
9481,AAPL.US,D,2022-04-20,0,168.76,168.8800,166.10,167.23,67929814,0,Wednesday
9482,AAPL.US,D,2022-04-21,0,168.91,171.5300,165.91,166.42,87227768,0,Thursday
9483,AAPL.US,D,2022-04-22,0,166.46,167.8699,161.50,161.79,84753396,0,Friday


In [9]:
apple['weekday'] = apple['DATE'].dt.weekday   # Mon=0 ... Sun=6
apple['week']    = apple['DATE'].dt.to_period('W-SUN')
apple.head()    


,TICKER,PER,DATE,TIME,OPEN,HIGH,LOW,CLOSE,VOL,OPENINT,DayOfWeek,weekday,week
0,AAPL.US,D,1984-09-07,0,0.10122,0.10246,0.10000,0.10122,97236149,0,Friday,4,1984-09-03/1984-09-09
1,AAPL.US,D,1984-09-10,0,0.10122,0.10153,0.09878,0.10062,75471114,0,Monday,0,1984-09-10/1984-09-16
2,AAPL.US,D,1984-09-11,0,0.10153,0.10428,0.10153,0.10246,177965367,0,Tuesday,1,1984-09-10/1984-09-16
3,AAPL.US,D,1984-09-12,0,0.10246,0.10306,0.09938,0.09938,155467926,0,Wednesday,2,1984-09-10/1984-09-16
4,AAPL.US,D,1984-09-13,0,0.10490,0.10520,0.10490,0.10490,242135546,0,Thursday,3,1984-09-10/1984-09-16


In [10]:
df1 = apple.copy()  

In [11]:
df1

,TICKER,PER,DATE,TIME,OPEN,HIGH,LOW,CLOSE,VOL,OPENINT,DayOfWeek,weekday,week
0,AAPL.US,D,1984-09-07,0,0.10122,0.10246,0.10000,0.10122,97236149,0,Friday,4,1984-09-03/1984-09-09
1,AAPL.US,D,1984-09-10,0,0.10122,0.10153,0.09878,0.10062,75471114,0,Monday,0,1984-09-10/1984-09-16
2,AAPL.US,D,1984-09-11,0,0.10153,0.10428,0.10153,0.10246,177965367,0,Tuesday,1,1984-09-10/1984-09-16
3,AAPL.US,D,1984-09-12,0,0.10246,0.10306,0.09938,0.09938,155467926,0,Wednesday,2,1984-09-10/1984-09-16
4,AAPL.US,D,1984-09-13,0,0.10490,0.10520,0.10490,0.10490,242135546,0,Thursday,3,1984-09-10/1984-09-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9479,AAPL.US,D,2022-04-18,0,163.92000,166.59840,163.57000,165.07000,69023941,0,Monday,0,2022-04-18/2022-04-24
9480,AAPL.US,D,2022-04-19,0,165.02000,167.82000,163.91000,167.40000,67723833,0,Tuesday,1,2022-04-18/2022-04-24
9481,AAPL.US,D,2022-04-20,0,168.76000,168.88000,166.10000,167.23000,67929814,0,Wednesday,2,2022-04-18/2022-04-24
9482,AAPL.US,D,2022-04-21,0,168.91000,171.53000,165.91000,166.42000,87227768,0,Thursday,3,2022-04-18/2022-04-24


In [12]:
df = df.rename(columns={"date_yyyymmdd": "DATE"})
df

,DATE,weekday
0,19840907,Friday
1,19840908,Saturday
2,19840909,Sunday
3,19840910,Monday
4,19840911,Tuesday
...,...,...
13737,20220418,Monday
13738,20220419,Tuesday
13739,20220420,Wednesday
13740,20220421,Thursday


In [13]:
df['DATE'] = pd.to_datetime(df['DATE'], format="%Y%m%d")
df.dtypes
df.head()

,DATE,weekday
0,1984-09-07,Friday
1,1984-09-08,Saturday
2,1984-09-09,Sunday
3,1984-09-10,Monday
4,1984-09-11,Tuesday


In [15]:
merged = pd.merge(
    df,   # this stays as left
    df1[['DATE', 'OPEN', 'CLOSE', 'VOL', 'week','HIGH','LOW']],  # this is right
    on='DATE',
    how='left'
)

merged 


,DATE,weekday,OPEN,CLOSE,VOL,week,HIGH,LOW
0,1984-09-07,Friday,0.10122,0.10122,97236149.0,1984-09-03/1984-09-09,0.10246,0.10000
1,1984-09-08,Saturday,NaN,NaN,NaN,NaT,NaN,NaN
2,1984-09-09,Sunday,NaN,NaN,NaN,NaT,NaN,NaN
3,1984-09-10,Monday,0.10122,0.10062,75471114.0,1984-09-10/1984-09-16,0.10153,0.09878
4,1984-09-11,Tuesday,0.10153,0.10246,177965367.0,1984-09-10/1984-09-16,0.10428,0.10153
...,...,...,...,...,...,...,...,...
13737,2022-04-18,Monday,163.92000,165.07000,69023941.0,2022-04-18/2022-04-24,166.59840,163.57000
13738,2022-04-19,Tuesday,165.02000,167.40000,67723833.0,2022-04-18/2022-04-24,167.82000,163.91000
13739,2022-04-20,Wednesday,168.76000,167.23000,67929814.0,2022-04-18/2022-04-24,168.88000,166.10000
13740,2022-04-21,Thursday,168.91000,166.42000,87227768.0,2022-04-18/2022-04-24,171.53000,165.91000


In [16]:
merged = merged[~merged['weekday'].isin(['Saturday', 'Sunday'])].reset_index(drop=True)
merged

,DATE,weekday,OPEN,CLOSE,VOL,week,HIGH,LOW
0,1984-09-07,Friday,0.10122,0.10122,97236149.0,1984-09-03/1984-09-09,0.10246,0.10000
1,1984-09-10,Monday,0.10122,0.10062,75471114.0,1984-09-10/1984-09-16,0.10153,0.09878
2,1984-09-11,Tuesday,0.10153,0.10246,177965367.0,1984-09-10/1984-09-16,0.10428,0.10153
3,1984-09-12,Wednesday,0.10246,0.09938,155467926.0,1984-09-10/1984-09-16,0.10306,0.09938
4,1984-09-13,Thursday,0.10490,0.10490,242135546.0,1984-09-10/1984-09-16,0.10520,0.10490
...,...,...,...,...,...,...,...,...
9811,2022-04-18,Monday,163.92000,165.07000,69023941.0,2022-04-18/2022-04-24,166.59840,163.57000
9812,2022-04-19,Tuesday,165.02000,167.40000,67723833.0,2022-04-18/2022-04-24,167.82000,163.91000
9813,2022-04-20,Wednesday,168.76000,167.23000,67929814.0,2022-04-18/2022-04-24,168.88000,166.10000
9814,2022-04-21,Thursday,168.91000,166.42000,87227768.0,2022-04-18/2022-04-24,171.53000,165.91000


In [17]:
# Count nulls in each column
print(merged.isnull().sum())

DATE         0
weekday      0
OPEN       332
CLOSE      332
VOL        332
week       332
HIGH       332
LOW        332
dtype: int64


In [18]:
# Or see rows that have any nulls
null_rows = merged[merged.isnull().any(axis=1)]
null_rows

,DATE,weekday,OPEN,CLOSE,VOL,week,HIGH,LOW
54,1984-11-22,Thursday,NaN,NaN,NaN,NaT,NaN,NaN
77,1984-12-25,Tuesday,NaN,NaN,NaN,NaT,NaN,NaN
82,1985-01-01,Tuesday,NaN,NaN,NaN,NaT,NaN,NaN
116,1985-02-18,Monday,NaN,NaN,NaN,NaT,NaN,NaN
150,1985-04-05,Friday,NaN,NaN,NaN,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...
9709,2021-11-25,Thursday,NaN,NaN,NaN,NaT,NaN,NaN
9730,2021-12-24,Friday,NaN,NaN,NaN,NaT,NaN,NaN
9746,2022-01-17,Monday,NaN,NaN,NaN,NaT,NaN,NaN
9771,2022-02-21,Monday,NaN,NaN,NaN,NaT,NaN,NaN


In [20]:
# Boolean mask for rows with any nulls
null_mask = merged.isnull().any(axis=1)

# Shift mask down by 1 and check where both current & previous are True
consec_nulls = null_mask & null_mask.shift(1, fill_value=False)

# Rows where consecutive nulls occur
consec_null_rows = merged[consec_nulls]

print("Any consecutive nulls? ->", consec_nulls.any())
print(consec_null_rows.head())


Any consecutive nulls? -> True
           DATE    weekday  OPEN  CLOSE  VOL week  HIGH  LOW
4438 2001-09-12  Wednesday   NaN    NaN  NaN  NaT   NaN  NaN
4439 2001-09-13   Thursday   NaN    NaN  NaN  NaT   NaN  NaN
4440 2001-09-14     Friday   NaN    NaN  NaN  NaT   NaN  NaN
5822 2007-01-02    Tuesday   NaN    NaN  NaN  NaT   NaN  NaN
7342 2012-10-30    Tuesday   NaN    NaN  NaN  NaT   NaN  NaN


In [21]:
target_nulls = null_rows[null_rows['weekday'].isin(['Monday','Tuesday','Wednesday'])]

# Count them
count_target_nulls = len(target_nulls)

print("Number of rows with nulls on Monday, Tuesday, or Wednesday:", count_target_nulls)

Number of rows with nulls on Monday, Tuesday, or Wednesday: 209


3.36% out of total are null in the dateset.
209/332 are from friday monday or tuesday.

In [22]:
# Ensure week column is string first
weeks_str = merged['week'].astype(str)

# Extract the start date (before the slash)
weeks_start = weeks_str.str.split('/').str[0]

# Convert to datetime
weeks_present = pd.to_datetime(weeks_start)

# Build full range of Mondays
full_range = pd.date_range(start=weeks_present.min(),
                           end=weeks_present.max(),
                           freq='W-MON')

# Find missing
missing_weeks = sorted(set(full_range) - set(weeks_present))

print("Total expected weeks:", len(full_range))
print("Weeks present:", len(weeks_present.unique()))
print("Missing weeks:", len(missing_weeks))
print("Some missing weeks:", missing_weeks[:10])


Total expected weeks: 1964
Weeks present: 1965
Missing weeks: 0
Some missing weeks: []


In [23]:
merged = merged.drop(columns=['week'])
merged

,DATE,weekday,OPEN,CLOSE,VOL,HIGH,LOW
0,1984-09-07,Friday,0.10122,0.10122,97236149.0,0.10246,0.10000
1,1984-09-10,Monday,0.10122,0.10062,75471114.0,0.10153,0.09878
2,1984-09-11,Tuesday,0.10153,0.10246,177965367.0,0.10428,0.10153
3,1984-09-12,Wednesday,0.10246,0.09938,155467926.0,0.10306,0.09938
4,1984-09-13,Thursday,0.10490,0.10490,242135546.0,0.10520,0.10490
...,...,...,...,...,...,...,...
9811,2022-04-18,Monday,163.92000,165.07000,69023941.0,166.59840,163.57000
9812,2022-04-19,Tuesday,165.02000,167.40000,67723833.0,167.82000,163.91000
9813,2022-04-20,Wednesday,168.76000,167.23000,67929814.0,168.88000,166.10000
9814,2022-04-21,Thursday,168.91000,166.42000,87227768.0,171.53000,165.91000


In [24]:
# Impute missing values using only past data
merged_impute = merged.fillna(method='ffill')
merged_impute


,DATE,weekday,OPEN,CLOSE,VOL,HIGH,LOW
0,1984-09-07,Friday,0.10122,0.10122,97236149.0,0.10246,0.10000
1,1984-09-10,Monday,0.10122,0.10062,75471114.0,0.10153,0.09878
2,1984-09-11,Tuesday,0.10153,0.10246,177965367.0,0.10428,0.10153
3,1984-09-12,Wednesday,0.10246,0.09938,155467926.0,0.10306,0.09938
4,1984-09-13,Thursday,0.10490,0.10490,242135546.0,0.10520,0.10490
...,...,...,...,...,...,...,...
9811,2022-04-18,Monday,163.92000,165.07000,69023941.0,166.59840,163.57000
9812,2022-04-19,Tuesday,165.02000,167.40000,67723833.0,167.82000,163.91000
9813,2022-04-20,Wednesday,168.76000,167.23000,67929814.0,168.88000,166.10000
9814,2022-04-21,Thursday,168.91000,166.42000,87227768.0,171.53000,165.91000


In [25]:
# Make sure DATE is datetime
merged['DATE'] = pd.to_datetime(merged['DATE'], format="%Y%m%d")

# Example: search rows between 2001-01-01 and 2001-12-31
start = "1984-11-18"
end   = "1984-11-25"

mask = (merged['DATE'] >= start) & (merged['DATE'] <= end)
subset = merged.loc[mask]

subset.head()


,DATE,weekday,OPEN,CLOSE,VOL,HIGH,LOW
51,1984-11-19,Monday,0.08870,0.08348,272088493.0,0.08931,0.08348
52,1984-11-20,Tuesday,0.08623,0.08623,307430447.0,0.08654,0.08623
53,1984-11-21,Wednesday,0.08809,0.08809,208729365.0,0.08870,0.08809
54,1984-11-22,Thursday,NaN,NaN,NaN,NaN,NaN
55,1984-11-23,Friday,0.08931,0.09052,160098193.0,0.09174,0.08931


In [26]:
# Make sure DATE is datetime
merged_impute['DATE'] = pd.to_datetime(merged_impute['DATE'], format="%Y%m%d")

# Example: search rows between 2001-01-01 and 2001-12-31
start = "1984-11-18"
end   = "1984-11-25"

mask_1 = (merged_impute['DATE'] >= start) & (merged_impute['DATE'] <= end)
subset1 = merged_impute.loc[mask_1]

subset1.head()


,DATE,weekday,OPEN,CLOSE,VOL,HIGH,LOW
51,1984-11-19,Monday,0.08870,0.08348,272088493.0,0.08931,0.08348
52,1984-11-20,Tuesday,0.08623,0.08623,307430447.0,0.08654,0.08623
53,1984-11-21,Wednesday,0.08809,0.08809,208729365.0,0.08870,0.08809
54,1984-11-22,Thursday,0.08809,0.08809,208729365.0,0.08870,0.08809
55,1984-11-23,Friday,0.08931,0.09052,160098193.0,0.09174,0.08931


In [27]:
merged_impute.to_csv(r"D:\data\notebooks\week-10\cleaned_apple_high_low.csv", index=False)

In [54]:
merged_impute.to_csv(r"D:\data\notebooks\week-4\weekly_cleaned_apple.csv", index=False)